In [28]:
from typing import List,Optional
import numpy as np
from utils.evaluate_rag import *
from utils.helper_functions import *
import os
import sys
from dotenv import load_dotenv
from langchain_core.documents import Document
from rank_bm25 import BM25Okapi

load_dotenv(dotenv_path="/Users/nilasark/advanced/.env")

path="/Users/nilasark/advanced/data/hyde_rag.pdf"

In [26]:
def encode_pdf_and_split_documents(path,chunk_size:int|None=None,chunk_overlap:int|None=None):
    from langchain_community.document_loaders import PyPDFLoader
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from langchain_community.vectorstores import Chroma
    from langchain_openai import OpenAIEmbeddings

    embeddings=OpenAIEmbeddings(model="text-embedding-3-small")
    chunker=RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        separators=["\n\n", "\n", " ", ""]
    )
    loader=PyPDFLoader(file_path="/Users/nilasark/advanced/data/hyde_rag.pdf")
    loaded_documents=loader.load()
    chunks=chunker.split_documents(loaded_documents)
    cleaned_text=replace_tab_with_space(chunks)
    vectorstore=Chroma.from_documents(cleaned_text,embeddings)

    return vectorstore,cleaned_text


In [5]:
vectorstore,clened_text=encode_pdf_and_split_documents(path)

Create BM25 index to retrieve document by keyword

In [16]:
corpus = [
    "Hello there good man!",
    "It is quite windy in London",
    "How is the weather today?"
]

tokenized_corpus=[doc.split(" ") for doc in corpus]
bm25=BM25Okapi(tokenized_corpus)

query="windy London"
tokenized_query=query.split(" ")
bm25_scores=bm25.get_scores(tokenized_query)
doc_top_n=bm25.get_top_n(tokenized_query,corpus,n=2)
print(doc_top_n)

['It is quite windy in London', 'How is the weather today?']


In [24]:

def create_bm25_index(documents:List[Document])->BM25Okapi:
    tokenized_documents=[doc.page_content.split() for doc in documents]
    return BM25Okapi(tokenized_documents)


In [25]:
bm25=create_bm25_index(clened_text)

In [29]:
def fusion_retrieval(vectorstore,bm25,query:str,k:int=5,alpha:float=0.5)->List[Document]:
    epsilon=1e-8

    all_docs=vectorstore.similarity_search("",k=vectorstore._collection.count())
    bm25_scores=bm25.get_scores(query.split(" "))

    vector_results=vectorstore.similarity_search_with_score(query,k=len(all_docs))
    vector_scores=np.array([scores for _,scores in vector_results])
    # Because the two scores have opposite meanings in this code.
    # For BM25:
    # Higher score = better match
    # For the FAISS vector score being used here:
    # Lower distance = better match
    # So the vector score needs to be flipped with 1 - ... so that after normalization, higher = better for both systems.

    normalised_vectorscores=1-((vector_scores-np.min(vector_scores))/(np.max(vector_scores)-np.min(vector_scores)+epsilon))
    normalised_bm25score=(bm25_scores-np.min(bm25_scores))/(np.max(bm25_scores)-np.min(bm25_scores)+epsilon)

    combined_scores=(alpha*normalised_vectorscores)+((1-alpha)*normalised_bm25score)
    sorted_scores=np.argsort(combined_scores)[::-1]
    return [all_docs[i] for i in sorted_scores[:k]]



In [32]:
query = "What happens when HyDE uses smaller instruction models like FLAN-T5 or Cohere?"
top_docs=fusion_retrieval(vectorstore,bm25,query,5,0.2)
doc_content=[doc.page_content for doc in top_docs]
show_context(doc_content)

Context:1
Krueger, Michael Petrov, Heidy Khlaaf, Girish Sas-
try, Pamela Mishkin, Brooke Chan, Scott Gray,
Nick Ryder, Mikhail Pavlov, Alethea Power, Lukasz
Kaiser, Mohammad Bavarian, Clemens Winter,
Philippe Tillet, Felipe Petroski Such, Dave Cum-
mings, Matthias Plappert, Fotios Chantzis, Eliza-
beth Barnes, Ariel Herbert-V oss, William Hebgen
Guss, Alex Nichol, Alex Paino, Nikolas Tezak, Jie
Tang, Igor Babuschkin, Suchir Balaji, Shantanu Jain,
William Saunders, Christopher Hesse, Andrew N.
Carr, Jan Leike, Josh Achiam, Vedant Misra, Evan
Morikawa, Alec Radford, Matthew Knight, Miles
Brundage, Mira Murati, Katie Mayer, Peter Welin-
der, Bob McGrew, Dario Amodei, Sam McCandlish,
Ilya Sutskever, and Wojciech Zaremba. 2021. Eval-
uating large language models trained on code.
Aakanksha Chowdhery, Sharan Narang, Jacob Devlin,
Maarten Bosma, Gaurav Mishra, Adam Roberts,
Paul Barham, Hyung Won Chung, Charles Sutton,
Sebastian Gehrmann, Parker Schuh, Kensen Shi,


Context:2
and Arnold Overwi

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from langchain_classic.chains import RetrievalQA
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever

alpha=0.5
doc_path=PyPDFLoader("/Users/nilasark/advanced/data/hyde_rag.pdf")
loaded_documents=doc_path.load()
chunker=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]
)
chunks=chunker.split_documents(loaded_documents)
embeddings=OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore=Chroma.from_documents(chunks,embeddings)
dense_vectors=vectorstore.as_retriever( search_kwargs={"k": 3})

sparse_vector=BM25Retriever.from_documents(chunks)
sparse_vector.k=3

hybrid_retriever=EnsembleRetriever(
    retrievers=[dense_vectors,sparse_vector],
    weights=[alpha,1-alpha]
)
llm=ChatOpenAI(model="gpt-4o-mini")
retrieval_chain=RetrievalQA.from_chain_type(
    llm=llm,retriever=hybrid_retriever,return_source_documents=True
)

query="What is the main topic of this document?"
result=retrieval_chain.invoke({"query":query})
print(f"Answer:{result["result"]}")
print("\n Sources")
for doc in result["source_documents"]:
    print(f"- {doc.page_content}......")

Answer:The main topic of the document is the HyDE model, which focuses on improving dense retrieval methods by using a generative language model and a contrastive encoder to assess relevance in document retrieval tasks. The paper discusses how HyDE captures relevance and instruction understanding, potentially eliminating the need for traditional relevance labels in information retrieval.

 Sources
- Figure 1: An illustration of the HyDE model. Documents snippets are shown. HyDE serves all types of queries
without changing the underlyingGPT-3 andContriever/mContriever models.
to human intent to follow instructions.
With these ingredients, we propose to
pivot through Hypothetical Document
Embeddings ( HyDE), and decompose dense
retrieval into two tasks, a generative task per-
formed by an instruction-following language
model and a document-document similarity task
performed by a contrastive encoder (Figure 1).
First, we feed the query to the generative model
and instruct it to "write a d